# 13 MoE_Routing_And_Experts

## Problem

MoE 为什么能把模型做大而不让每个 token 都激活全部参数？router、expert、top-k routing 分别在做什么？

## Dependencies

建议先掌握 FFN 的作用，因为 MoE 常常可以理解成“把一个 FFN 换成很多专家 FFN”。


## Module_Goals

- 理解 dense FFN 与 MoE FFN 的关键区别
- 理解 router 如何给不同 token 选专家
- 理解 top-k routing 与 activated parameters 的概念
- 会写一个最小可运行的 MoE 层

## Scope_And_Approach

这个 notebook 会先用直觉和小例子说明问题，再给出最小实现。重点是把模块为什么存在、输入输出是什么、关键步骤如何衔接说明清楚。


## 直觉解释

普通 dense FFN 的做法是：每个 token 都走同一套参数。

MoE 的想法是：

- 准备很多个专家，每个专家擅长的模式可能不同。
- 对每个 token，不必把所有专家都跑一遍。
- router 先判断“这个 token 更该送给谁”。
- 只激活 top-k 个专家，让它们输出加权结果。

这像一个分诊系统：不是所有病人都去看所有科室，而是先分流到更合适的少数专家。这样总参数可以很大，但单次激活成本不一定跟着同比例增长。


In [ ]:
import numpy as np

np.random.seed(99)
np.set_printoptions(precision=3, suppress=True)

x = np.array([
    [0.2, 0.5, 0.1, 0.7],
    [0.9, 0.3, 0.4, 0.2],
    [0.1, 0.8, 0.6, 0.5],
])  # (num_tokens, hidden_dim)

num_experts = 3
hidden_dim = 4
ffn_dim = 6

def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)


In [ ]:
router_w = np.random.randn(hidden_dim, num_experts) * 0.3
router_logits = x @ router_w
router_probs = softmax(router_logits)
print('router_probs =')
print(router_probs)

experts = []
for _ in range(num_experts):
    w1 = np.random.randn(hidden_dim, ffn_dim) * 0.3
    w2 = np.random.randn(ffn_dim, hidden_dim) * 0.3
    experts.append((w1, w2))

outputs = []
for token_idx, token in enumerate(x):
    probs = router_probs[token_idx]
    top_k = probs.argsort()[-2:][::-1]
    combined = np.zeros(hidden_dim)
    print(f'token {token_idx} -> experts {top_k}')
    for expert_id in top_k:
        w1, w2 = experts[expert_id]
        hidden = np.maximum(token @ w1, 0)
        expert_out = hidden @ w2
        combined += probs[expert_id] * expert_out
    outputs.append(combined)

outputs = np.stack(outputs)
print('moe outputs =')
print(outputs)


## Trade_Offs_And_Risk_Points

- 把 MoE 想成“所有专家都跑一遍再平均”。关键恰恰在于只激活少数专家。
- 只关注总参数量，而忽略 activated parameters 才更接近一次前向真正用到的参数。
- 忽视路由平衡问题。如果所有 token 都挤向少数专家，MoE 会训练得很别扭。

## Checkpoints

- 把 top-k 从 2 改成 1 或 3，想想容量和计算量会怎么变。
- 观察不同 token 被分到的专家是否一致。
- 思考为什么 MoE 通常需要额外的 load balancing 约束。

## Summary

MoE 的核心价值是把“总容量”和“单次激活成本”部分解耦。这也是为什么许多大模型会走稠密参数与稀疏激活结合的路线。

## Next_Module

下一节开始进入训练目标。先看预训练阶段到底在优化什么。
